# 00 · Cell control
Hands-on control of the cell from a notebook: arms, bed, extruders. Each part is one cell you run on its own.
Run **Connect** first. **Disconnect** is the last cell. If anything goes wrong, run the **STOP** cell.

Everything here calls the same three drivers the test scripts use (`src/cell/hardware/arm.py`, `turntable.py`, `extruder.py`) through `cell.runs.cylinder.Cell`, so what you learn here carries straight into the test notebooks.

Set `SIM = True` to drive cell studio's simulators instead of the hardware.

## Connect

In [ ]:
import sys, os, time, math
ROOT = os.path.abspath("..")                      # repo root, when this notebook runs from notebooks/
sys.path[:0] = [ROOT, os.path.join(ROOT, "src"), os.path.join(ROOT, "ui")]   # not needed after: pip install -e .

from cell.runs.cylinder import Cell, DEFAULTS, SIDE, PARK, SAFE_Z, NOZZLE

SIM = False                                        # True: cell studio simulators, no hardware
ARMS = [0, 1]                                      # 0 = right, 1 = left

g = {"arms": {tl: dict(DEFAULTS[tl], T=tl, side=SIDE[tl]) for tl in ARMS}}
cell = Cell(g, SIM)
print("connected:", ", ".join(SIDE[t] for t in ARMS), "| bed angle %.2f deg" % cell.tt.get_angle())

## Where is everything?
Pose is X, Y, Z, roll, pitch, yaw in each arm's own frame (mm, deg). Z0 is the height at which that nozzle touches the disc;
never command below it.

In [ ]:
for tl in ARMS:
    p = cell.pose(tl)
    print(f"{SIDE[tl]:5}: X {p[0]:7.2f}  Y {p[1]:7.2f}  Z {p[2]:7.2f}   (Z0 {g['arms'][tl]['Z0']:.2f}, nozzle {NOZZLE[tl]:g} mm)")
print(f"bed  : {cell.tt.get_angle():8.2f} deg")
if not SIM:
    for tl in ARMS:
        print(f"{SIDE[tl]:5} hot end: {cell.hot_end_temp(tl):.1f} C")

## Jog an arm
`jog(side, dx, dy, dz)` moves relative to where the arm is now, keeping the nozzle orientation. Small numbers first.
The Z guard refuses anything below that arm's Z0.

In [ ]:
def jog(tl, dx=0.0, dy=0.0, dz=0.0, speed=20):
    x, y, z = cell.pose(tl)[:3]
    z_new = z + dz
    if z_new < g["arms"][tl]["Z0"] - 1e-6:
        raise ValueError(f"{SIDE[tl]} Z {z_new:.2f} would be below Z0 {g['arms'][tl]['Z0']:.2f}")
    cell.move(tl, x + dx, y + dy, z_new, speed=speed, wait=True)
    print(f"{SIDE[tl]}: now X {x+dx:.2f} Y {y+dy:.2f} Z {z_new:.2f}")

def goto(tl, x, y, z, speed=20):
    if z < g["arms"][tl]["Z0"] - 1e-6:
        raise ValueError(f"{SIDE[tl]} Z {z:.2f} would be below Z0 {g['arms'][tl]['Z0']:.2f}")
    cell.move(tl, x, y, z, speed=speed, wait=True)

def park(tl, speed=60):
    x, y, z = cell.pose(tl)[:3]
    cell.move(tl, x, y, max(z, SAFE_Z), speed=speed, wait=True)      # up first
    cell.move(tl, *PARK, speed=speed, wait=True)
    print(f"{SIDE[tl]}: parked")

# examples (uncomment one):
# jog(0, dz=+5)
# jog(0, dx=-2, dy=+2)
# goto(0, 480, -30, 130)
# park(0)

## The bed
Angle is continuous (it does not wrap at 360). `rotate_velocity` runs at a speed until `stop_rotation`;
`rotate_relative` makes one planned move with the controller's own ramp. Positive speed turns the disc clockwise seen from above.

In [ ]:
def bed_run(deg_per_s):            cell.tt.rotate_velocity(deg_per_s)
def bed_stop():                    cell.tt.stop_rotation()
def bed_by(deg, deg_per_s=20):     cell.tt.rotate_relative(deg, deg_per_s, wait=True); print("bed at %.2f" % cell.tt.get_angle())
def bed_angle():                   return cell.tt.get_angle()

# examples:
# bed_by(90)            # a quarter turn
# bed_run(10); time.sleep(3); bed_stop()
print("bed at %.2f deg" % bed_angle())

## Extruders
The Klipper macros: `EXTRUDE_ON T=<0|1> RATE=<mm/s> TIMEOUT=5` runs filament continuously (re-send within 5 s or the watchdog stops it),
`EXTRUDE_OFF [T=]` stops, `EXTRUDE_STATUS` reports. `EXTRUDE_SCALE` is sticky; keep it at 100 unless you mean it.
Heating uses `extrude.py`'s path (M104/M109 on the test config's heaters).

In [ ]:
def heat(tl, temp_c):            cell.ext.set_temperature(tl, temp_c)                     # does not wait
def heat_wait(tl, temp_c):       cell.ext.heat_and_wait(tl, temp_c)
def temps():                     return {SIDE[t]: cell.hot_end_temp(t) for t in ARMS}
def extrude_on(tl, rate_mm_s):   cell.gcode(f"EXTRUDE_ON T={tl} RATE={rate_mm_s:.3f} TIMEOUT=5")
def extrude_off(tl=None):        cell.gcode("EXTRUDE_OFF" if tl is None else f"EXTRUDE_OFF T={tl}")
def extrude_status():            return cell.gcode("EXTRUDE_STATUS")
def purge(tl, mm=20, rate=3.0):
    extrude_on(tl, rate)
    time.sleep(mm / rate)
    extrude_off(tl)

for tl in (0, 1):
    cell.gcode(f"EXTRUDE_SCALE T={tl} PERCENT=100")
# examples:
# heat(0, 215); print(temps())
# purge(0, mm=30)
# extrude_status()

## STOP (run this if anything looks wrong)

In [ ]:
def stop_everything():
    try: cell.gcode("EXTRUDE_OFF")
    except Exception as e: print("extruders:", e)
    try: cell.tt.stop_rotation()
    except Exception as e: print("bed:", e)
    for tl in ARMS:
        try:
            x, y, z = cell.pose(tl)[:3]
            cell.move(tl, x, y, z + 10, speed=20, wait=True)       # lift 10 mm, nothing else
        except Exception as e: print(SIDE[tl], e)
    print("stopped: extruders off, bed stopped, arms lifted 10 mm")

# stop_everything()

## Disconnect

In [ ]:
cell.stop_keep.set()
for a in cell.arms.values():
    a.disconnect()
cell.tt.disconnect()
print("disconnected")